# 🛍️ Retail Data — Exploratory Data Analysis

**Schema overview:** `products`, `customers`, `orders`, `order_items`, `promotions`, `payments`, `shipments`, `returns`, `reviews`, `inventory`, `web_traffic`, `geography`, `sales`

---
### Table of Contents
1. [Setup & Data Loading](#1-setup)
2. [Schema Validation & Data Quality](#2-quality)
3. [Customer Analysis](#3-customers)
4. [Product & Catalog Analysis](#4-products)
5. [Orders & Revenue Analysis](#5-orders)
6. [Promotions & Discounts](#6-promos)
7. [Payments & Shipping](#7-payments)
8. [Returns & Reviews](#8-returns)
9. [Inventory Health](#9-inventory)
10. [Web Traffic](#10-traffic)
11. [Geographic Analysis](#11-geo)
12. [Key Takeaways](#12-summary)

## 1. Setup & Data Loading <a id='1-setup'></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ── Aesthetics ──────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
PALETTE  = sns.color_palette('muted')
FIG_W, FIG_H = 14, 5
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False,
                     'axes.spines.right': False})

print('Libraries loaded ✔')

In [ ]:
# ── Load all tables ──────────────────────────────────────────────────────────
# Replace the paths / read calls with whatever source you use
# (CSV, Parquet, SQL connection, etc.).

def load(name, parse_dates=None):
    """Thin wrapper – swap this out for your actual data source."""
    return pd.read_csv(f'{name}.csv', parse_dates=parse_dates, low_memory=False)

products   = load('products')
customers  = load('customers',  parse_dates=['signup_date'])
orders     = load('orders',     parse_dates=['order_date'])
order_items= load('order_items')
promotions = load('promotions', parse_dates=['start_date','end_date'])
payments   = load('payments')
shipments  = load('shipments',  parse_dates=['ship_date','delivery_date'])
returns    = load('returns',    parse_dates=['return_date'])
reviews    = load('reviews',    parse_dates=['review_date'])
inventory  = load('inventory',  parse_dates=['snapshot_date'])
web_traffic= load('web_traffic',parse_dates=['date'])
geography  = load('geography')
sales      = load('sales',      parse_dates=['Date'])

TABLES = {
    'products': products, 'customers': customers, 'orders': orders,
    'order_items': order_items, 'promotions': promotions, 'payments': payments,
    'shipments': shipments, 'returns': returns, 'reviews': reviews,
    'inventory': inventory, 'web_traffic': web_traffic,
    'geography': geography, 'sales': sales
}

for name, df in TABLES.items():
    print(f'{name:<15} {df.shape[0]:>8,} rows  ×  {df.shape[1]:>3} cols')

## 2. Schema Validation & Data Quality <a id='2-quality'></a>

In [ ]:
# ── Missing-value heatmap ─────────────────────────────────────────────────
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.flatten()

for ax, (name, df) in zip(axes, TABLES.items()):
    miss = df.isnull().mean().sort_values(ascending=False)
    miss = miss[miss > 0]
    if miss.empty:
        ax.text(0.5, 0.5, 'No nulls', ha='center', va='center', fontsize=10)
    else:
        miss.plot.barh(ax=ax, color='steelblue')
        ax.xaxis.set_major_formatter(mticker.PercentFormatter(1))
    ax.set_title(name, fontweight='bold')

for ax in axes[len(TABLES):]:
    ax.set_visible(False)

plt.suptitle('Missing Values by Table', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Duplicate check ───────────────────────────────────────────────────────
PKS = {
    'products': 'product_id', 'customers': 'customer_id',
    'orders': 'order_id',     'promotions': 'promo_id',
    'returns': 'return_id',   'reviews': 'review_id',
    'geography': 'zip'
}

print('=== Primary-key uniqueness ===')
for tbl, pk in PKS.items():
    df = TABLES[tbl]
    n_dup = df[pk].duplicated().sum()
    flag = '✔' if n_dup == 0 else f'⚠ {n_dup:,} duplicates'
    print(f'  {tbl:<15} [{pk}]  {flag}')

# Composite PK for order_items
n_dup_oi = order_items.duplicated(subset=['order_id','product_id']).sum()
print(f'  order_items    [order_id+product_id]  {"✔" if n_dup_oi==0 else f"⚠ {n_dup_oi:,} dup"}')

In [ ]:
# ── Referential integrity spot-checks ────────────────────────────────────
checks = [
    (orders,      'customer_id', customers,  'customer_id', 'orders → customers'),
    (order_items, 'order_id',    orders,     'order_id',    'order_items → orders'),
    (order_items, 'product_id',  products,   'product_id',  'order_items → products'),
    (returns,     'order_id',    orders,     'order_id',    'returns → orders'),
    (reviews,     'order_id',    orders,     'order_id',    'reviews → orders'),
    (inventory,   'product_id',  products,   'product_id',  'inventory → products'),
]

print('=== Referential integrity ===')
for child, ck, parent, pk, label in checks:
    orphans = (~child[ck].isin(parent[pk])).sum()
    flag = '✔' if orphans == 0 else f'⚠ {orphans:,} orphan rows'
    print(f'  {label:<35} {flag}')

In [ ]:
# ── Date-range summary ────────────────────────────────────────────────────
date_cols = [
    (orders,      'order_date',     'orders'),
    (customers,   'signup_date',    'customers'),
    (shipments,   'ship_date',      'shipments'),
    (returns,     'return_date',    'returns'),
    (reviews,     'review_date',    'reviews'),
    (web_traffic, 'date',           'web_traffic'),
    (inventory,   'snapshot_date',  'inventory'),
    (sales,       'Date',           'sales'),
]

print(f'{"Table":<15} {"Column":<20} {"Min":<14} {"Max"}')
print('-' * 65)
for df, col, name in date_cols:
    print(f'{name:<15} {col:<20} {str(df[col].min())[:10]:<14} {str(df[col].max())[:10]}')

## 3. Customer Analysis <a id='3-customers'></a>

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(FIG_W, FIG_H))

# Signups over time
customers.set_index('signup_date').resample('ME')['customer_id'].count().plot(
    ax=axes[0], color=PALETTE[0])
axes[0].set_title('Monthly New Signups')
axes[0].set_xlabel('')

# Gender split
customers['gender'].value_counts().plot.pie(
    ax=axes[1], autopct='%1.1f%%', startangle=90,
    colors=PALETTE[:customers['gender'].nunique()])
axes[1].set_ylabel('')
axes[1].set_title('Gender Distribution')

# Acquisition channel
ch = customers['acquisition_channel'].value_counts()
ch.plot.barh(ax=axes[2], color=PALETTE)
axes[2].set_title('Acquisition Channel')
axes[2].set_xlabel('Customers')

plt.suptitle('Customer Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(FIG_W, FIG_H))

# Age group
age_order = sorted(customers['age_group'].dropna().unique())
customers['age_group'].value_counts().reindex(age_order).plot.bar(
    ax=axes[0], color=PALETTE[1], rot=0)
axes[0].set_title('Age Group Distribution')
axes[0].set_xlabel('Age Group')

# Orders per customer
opc = orders.groupby('customer_id').size()
opc.clip(upper=20).plot.hist(bins=20, ax=axes[1], color=PALETTE[2], edgecolor='white')
axes[1].set_title('Orders per Customer (capped @ 20)')
axes[1].set_xlabel('Order Count')

print(f'One-time buyers : {(opc == 1).mean():.1%}')
print(f'Repeat buyers   : {(opc  > 1).mean():.1%}')
print(f'Mean orders/cust: {opc.mean():.2f}')

plt.tight_layout()
plt.show()

## 4. Product & Catalog Analysis <a id='4-products'></a>

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(FIG_W, FIG_H))

# Category breakdown
products['category'].value_counts().plot.barh(ax=axes[0], color=PALETTE)
axes[0].set_title('SKUs by Category')

# Price distribution
products['price'].plot.hist(bins=40, ax=axes[1], color=PALETTE[0], edgecolor='white')
axes[1].set_title('Price Distribution')
axes[1].set_xlabel('Price')

# Gross-margin proxy
products['margin_pct'] = (products['price'] - products['cogs']) / products['price'] * 100
products['margin_pct'].plot.hist(bins=40, ax=axes[2], color=PALETTE[3], edgecolor='white')
axes[2].set_title('Gross Margin % Distribution')
axes[2].set_xlabel('Margin %')

plt.suptitle('Product Catalog', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(products[['price','cogs','margin_pct']].describe().round(2))

In [ ]:
# Top products by revenue
item_rev = (
    order_items
    .assign(revenue=lambda x: x['unit_price'] * x['quantity'] - x['discount_amount'])
    .groupby('product_id')['revenue'].sum()
    .reset_index()
    .merge(products[['product_id','product_name','category']], on='product_id')
    .sort_values('revenue', ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(FIG_W, FIG_H))

item_rev.head(15).set_index('product_name')['revenue'].plot.barh(
    ax=axes[0], color=PALETTE[0])
axes[0].invert_yaxis()
axes[0].set_title('Top 15 Products by Revenue')
axes[0].set_xlabel('Revenue')

# Revenue share by category
item_rev.groupby('category')['revenue'].sum().sort_values().plot.barh(
    ax=axes[1], color=PALETTE)
axes[1].set_title('Revenue by Category')

plt.tight_layout()
plt.show()

## 5. Orders & Revenue Analysis <a id='5-orders'></a>

In [ ]:
# Build enriched orders table
order_rev = (
    order_items
    .assign(line_rev=lambda x: x['unit_price'] * x['quantity'] - x['discount_amount'])
    .groupby('order_id').agg(
        revenue=('line_rev','sum'),
        items=('quantity','sum'),
        discount=('discount_amount','sum')
    )
    .reset_index()
)
orders_full = orders.merge(order_rev, on='order_id', how='left')

# Monthly revenue trend
monthly = (
    orders_full[orders_full['order_status'] != 'Cancelled']
    .set_index('order_date')
    .resample('ME')['revenue']
    .agg(['sum','count'])
)

fig, axes = plt.subplots(2, 1, figsize=(FIG_W, 8), sharex=True)

monthly['sum'].plot(ax=axes[0], color=PALETTE[0], linewidth=2)
axes[0].set_title('Monthly Revenue')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))

monthly['count'].plot(ax=axes[1], color=PALETTE[1], linewidth=2)
axes[1].set_title('Monthly Order Volume')

plt.suptitle('Revenue & Volume Trends', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(FIG_W, FIG_H))

# Order status
orders['order_status'].value_counts().plot.pie(
    ax=axes[0], autopct='%1.1f%%', startangle=90, colors=PALETTE)
axes[0].set_ylabel('')
axes[0].set_title('Order Status')

# Order source
orders['order_source'].value_counts().plot.bar(
    ax=axes[1], color=PALETTE, rot=25)
axes[1].set_title('Order Source')

# Device type
orders['device_type'].value_counts().plot.bar(
    ax=axes[2], color=PALETTE[2:], rot=0)
axes[2].set_title('Device Type')

plt.suptitle('Order Attributes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Average order value distribution
fig, axes = plt.subplots(1, 2, figsize=(FIG_W, FIG_H))

orders_full['revenue'].clip(upper=orders_full['revenue'].quantile(0.99)).plot.hist(
    bins=50, ax=axes[0], color=PALETTE[0], edgecolor='white')
axes[0].set_title('Order Value Distribution (99th pct cap)')
axes[0].set_xlabel('Revenue')

# Weekday pattern
orders['dow'] = orders['order_date'].dt.day_name()
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
orders['dow'].value_counts().reindex(dow_order).plot.bar(
    ax=axes[1], color=PALETTE, rot=30)
axes[1].set_title('Orders by Day of Week')

aov = orders_full['revenue'].mean()
print(f'Average Order Value (all): ${aov:,.2f}')

plt.tight_layout()
plt.show()

## 6. Promotions & Discounts <a id='6-promos'></a>

In [ ]:
# Promo usage rate
promo_usage = order_items.copy()
promo_usage['has_promo'] = promo_usage['promo_id'].notna() | promo_usage['promo_id_2'].notna()
promo_usage['is_stacked'] = promo_usage['promo_id'].notna() & promo_usage['promo_id_2'].notna()

print(f"Lines with ≥1 promo  : {promo_usage['has_promo'].mean():.1%}")
print(f"Lines with 2 promos  : {promo_usage['is_stacked'].mean():.1%}")

# Discount depth by promo type
promo_items = (
    order_items[order_items['promo_id'].notna()]
    .merge(promotions[['promo_id','promo_type','promo_name']], on='promo_id', how='left')
)
promo_items['disc_pct'] = promo_items['discount_amount'] / \
    (promo_items['unit_price'] * promo_items['quantity']) * 100

fig, axes = plt.subplots(1, 2, figsize=(FIG_W, FIG_H))

promo_items.groupby('promo_type')['disc_pct'].median().sort_values().plot.barh(
    ax=axes[0], color=PALETTE)
axes[0].set_title('Median Discount % by Promo Type')

# Top 10 promos by total discount
top_promos = promo_items.groupby('promo_name')['discount_amount'].sum().nlargest(10)
top_promos.plot.barh(ax=axes[1], color=PALETTE[1])
axes[1].invert_yaxis()
axes[1].set_title('Top 10 Promos by Total Discount $')

plt.tight_layout()
plt.show()

## 7. Payments & Shipping <a id='7-payments'></a>

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(FIG_W, FIG_H))

# Payment method split
payments['payment_method'].value_counts().plot.pie(
    ax=axes[0], autopct='%1.1f%%', startangle=90, colors=PALETTE)
axes[0].set_ylabel('')
axes[0].set_title('Payment Method')

# Installments distribution
payments['installments'].value_counts().sort_index().plot.bar(
    ax=axes[1], color=PALETTE[2], rot=0)
axes[1].set_title('Installment Frequency')
axes[1].set_xlabel('# Installments')

# Delivery time
shipments['delivery_days'] = (shipments['delivery_date'] - shipments['ship_date']).dt.days
shipments['delivery_days'].clip(lower=0, upper=30).plot.hist(
    bins=30, ax=axes[2], color=PALETTE[3], edgecolor='white')
axes[2].set_title('Delivery Days Distribution')
axes[2].set_xlabel('Days')

print(f"Median delivery days : {shipments['delivery_days'].median():.0f}")
print(f"Orders not shipped   : {orders.shape[0] - shipments.shape[0]:,}")

plt.suptitle('Payments & Shipping', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Returns & Reviews <a id='8-returns'></a>

In [ ]:
# Return rate
n_orders = orders.shape[0]
n_returns = returns['order_id'].nunique()
print(f'Return rate (order level): {n_returns/n_orders:.1%}')

fig, axes = plt.subplots(1, 3, figsize=(FIG_W, FIG_H))

# Return reasons
returns['return_reason'].value_counts().plot.barh(ax=axes[0], color=PALETTE)
axes[0].invert_yaxis()
axes[0].set_title('Return Reasons')

# Return rate by category
ret_prod = returns.merge(products[['product_id','category']], on='product_id')
sold_prod = order_items.merge(products[['product_id','category']], on='product_id')
ret_by_cat = ret_prod.groupby('category')['return_quantity'].sum() / \
             sold_prod.groupby('category')['quantity'].sum()
ret_by_cat.sort_values().plot.barh(ax=axes[1], color=PALETTE[1])
axes[1].xaxis.set_major_formatter(mticker.PercentFormatter(1))
axes[1].set_title('Return Rate by Category')

# Rating distribution
reviews['rating'].value_counts().sort_index().plot.bar(
    ax=axes[2], color=PALETTE[2], rot=0)
axes[2].set_title('Review Rating Distribution')
axes[2].set_xlabel('Rating')

print(f"Mean rating          : {reviews['rating'].mean():.2f} / 5")
print(f"Review coverage      : {reviews['order_id'].nunique()/n_orders:.1%} of orders")

plt.suptitle('Returns & Reviews', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Rating vs return rate correlation
prod_rating = reviews.groupby('product_id')['rating'].mean().rename('avg_rating')
prod_ret_rate = (
    returns.groupby('product_id')['return_quantity'].sum() /
    order_items.groupby('product_id')['quantity'].sum()
).rename('return_rate')

corr_df = pd.concat([prod_rating, prod_ret_rate], axis=1).dropna()

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(corr_df['avg_rating'], corr_df['return_rate'], alpha=0.4, color=PALETTE[0])
ax.set_xlabel('Average Rating')
ax.set_ylabel('Return Rate')
ax.set_title('Rating vs Return Rate (per product)')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1))

r = corr_df['avg_rating'].corr(corr_df['return_rate'])
ax.text(0.05, 0.92, f'Pearson r = {r:.2f}', transform=ax.transAxes)
plt.tight_layout()
plt.show()

## 9. Inventory Health <a id='9-inventory'></a>

In [ ]:
# Latest snapshot
latest_inv = inventory[inventory['snapshot_date'] == inventory['snapshot_date'].max()]

kpis = {
    'Stockout rate'   : f"{latest_inv['stockout_flag'].mean():.1%}",
    'Overstock rate'  : f"{latest_inv['overstock_flag'].mean():.1%}",
    'Reorder rate'    : f"{latest_inv['reorder_flag'].mean():.1%}",
    'Avg fill rate'   : f"{latest_inv['fill_rate'].mean():.1%}",
    'Avg DOS'         : f"{latest_inv['days_of_supply'].median():.1f} days",
    'Avg sell-through': f"{latest_inv['sell_through_rate'].mean():.1%}",
}
for k, v in kpis.items():
    print(f'{k:<22}: {v}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(FIG_W, FIG_H))

# Stockout trend
inv_monthly = inventory.groupby('snapshot_date')['stockout_flag'].mean()
inv_monthly.plot(ax=axes[0], color=PALETTE[3], linewidth=2)
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(1))
axes[0].set_title('Monthly Stockout Rate')
axes[0].set_xlabel('')

# Days of supply
latest_inv['days_of_supply'].clip(upper=90).plot.hist(
    bins=40, ax=axes[1], color=PALETTE[0], edgecolor='white')
axes[1].set_title('Days of Supply (capped @ 90)')
axes[1].set_xlabel('Days')

# Sell-through by product (top 20)
top_st = latest_inv.merge(products[['product_id','product_name']], on='product_id')\
                   .nlargest(15, 'sell_through_rate')\
                   .set_index('product_name')['sell_through_rate']
top_st.plot.barh(ax=axes[2], color=PALETTE[1])
axes[2].xaxis.set_major_formatter(mticker.PercentFormatter(1))
axes[2].set_title('Top 15 Sell-Through Rate')

plt.suptitle('Inventory Health', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Web Traffic <a id='10-traffic'></a>

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(FIG_W, 9))

wt_daily = web_traffic.set_index('date')

wt_daily['sessions'].resample('W').sum().plot(
    ax=axes[0,0], color=PALETTE[0], linewidth=1.5)
axes[0,0].set_title('Weekly Sessions')

wt_daily['bounce_rate'].resample('W').mean().plot(
    ax=axes[0,1], color=PALETTE[3], linewidth=1.5)
axes[0,1].yaxis.set_major_formatter(mticker.PercentFormatter(1))
axes[0,1].set_title('Weekly Avg Bounce Rate')

wt_daily['avg_session_duration_sec'].resample('W').mean().plot(
    ax=axes[1,0], color=PALETTE[2], linewidth=1.5)
axes[1,0].set_title('Weekly Avg Session Duration (sec)')

web_traffic.groupby('traffic_source')['sessions'].sum().sort_values().plot.barh(
    ax=axes[1,1], color=PALETTE)
axes[1,1].set_title('Sessions by Traffic Source')

plt.suptitle('Web Traffic Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Traffic to conversion rate (sessions vs orders on same day)
orders_daily = orders.groupby('order_date').size().rename('orders')
traffic_daily = web_traffic.set_index('date')['sessions']
tco = pd.concat([orders_daily, traffic_daily], axis=1).dropna()
tco['conv_rate'] = tco['orders'] / tco['sessions']

fig, ax = plt.subplots(figsize=(FIG_W, 4))
tco['conv_rate'].rolling(7).mean().plot(ax=ax, color=PALETTE[1], linewidth=2)
ax.set_title('7-day Rolling Traffic-to-Order Conversion Rate')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1))
plt.tight_layout()
plt.show()

## 11. Geographic Analysis <a id='11-geo'></a>

In [ ]:
# Enrich orders with region
orders_geo = orders.merge(geography[['zip','region','district']], on='zip', how='left')
orders_geo = orders_geo.merge(order_rev, on='order_id', how='left')

fig, axes = plt.subplots(1, 2, figsize=(FIG_W, FIG_H))

# Revenue by region
orders_geo.groupby('region')['revenue'].sum().sort_values().plot.barh(
    ax=axes[0], color=PALETTE)
axes[0].set_title('Revenue by Region')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e6:.1f}M'))

# Orders by district (top 20)
(
    orders_geo.groupby('district').size()
    .nlargest(20)
    .sort_values()
    .plot.barh(ax=axes[1], color=PALETTE[1])
)
axes[1].set_title('Top 20 Districts by Order Volume')

plt.suptitle('Geographic Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Customer density by region
cust_geo = customers.merge(geography[['zip','region']], on='zip', how='left')
region_stats = (
    cust_geo.groupby('region').agg(customers=('customer_id','nunique'))
    .join(
        orders_geo.groupby('region').agg(orders=('order_id','count'),
                                         revenue=('revenue','sum'))
    )
)
region_stats['aov'] = region_stats['revenue'] / region_stats['orders']
region_stats['orders_per_cust'] = region_stats['orders'] / region_stats['customers']

print(region_stats.sort_values('revenue', ascending=False).to_string())

## 12. Key Takeaways <a id='12-summary'></a>

In [ ]:
# ── High-level KPI card ───────────────────────────────────────────────────
total_rev     = order_rev['revenue'].sum()
total_orders  = orders.shape[0]
total_custs   = customers.shape[0]
aov           = total_rev / total_orders
avg_margin    = products['margin_pct'].mean()
return_rate   = returns['order_id'].nunique() / total_orders
avg_rating    = reviews['rating'].mean()
stockout_r    = inventory['stockout_flag'].mean()

print('='*50)
print('   RETAIL EDA — SUMMARY METRICS')
print('='*50)
print(f'  Total Revenue        : ${total_rev:>12,.0f}')
print(f'  Total Orders         : {total_orders:>12,}')
print(f'  Total Customers      : {total_custs:>12,}')
print(f'  Avg Order Value      : ${aov:>12,.2f}')
print(f'  Avg Gross Margin     : {avg_margin:>11.1f}%')
print(f'  Return Rate          : {return_rate:>11.1%}')
print(f'  Avg Product Rating   : {avg_rating:>11.2f} / 5')
print(f'  Avg Stockout Rate    : {stockout_r:>11.1%}')
print('='*50)

### 🔍 Suggested Next Steps
- **RFM Segmentation** — combine recency, frequency, monetary from `orders + order_items` to tier the customer base.
- **Promo Uplift Analysis** — compare revenue per order with vs without a promotion using matched pairs or regression.
- **Churn Prediction** — use `signup_date` and last `order_date` to define lapsed customers; build a binary classifier.
- **Inventory Forecasting** — use `inventory` + `order_items` to predict future `days_of_supply` per SKU.
- **Return-Reason Deep Dive** — cross-tab `return_reason × category × rating` to identify fixable quality issues.
- **Geo-Demographic LTV** — combine `geography.region`, `customers.age_group`, and order history for LTV modelling.